### Install requiremnts

In [7]:
!pip install -r ../requirements.txt

In [8]:
import sys
sys.path.append('../')
from checkpoints import DT, MT, LT
import os


from validate_birdset import load_model, get_test_loader, test
from checkpoints import DT, MT, LT
import numpy as np
import requests
import zipfile
import glob
import gdown

DOWN_TASKS = ["HSN"]
DEVICE = "cuda" # change to "cpu" if no gpu is available.

In [9]:
def run_testing(ckpts, extracted_interval=7, target_length=701, device='cuda'):
    for down_task in DOWN_TASKS:

        print(ckpts)

        models = []
        configs = []

        # ---------------------------------------------------------
        # Load models and their configs from checkpoints
        # ---------------------------------------------------------
        for ckpt in ckpts:
            m, cfg = load_model(ckpt, device=device)
            models.append(m)
            # Override validation parameters for evaluation
            cfg.frontend.val_target_length = target_length
            cfg.event_decoder.val.extracted_interval = extracted_interval
            # Set dataset name to current downstream task
            cfg.train.dataset_name = down_task
            configs.append(cfg)

        results = dict()
        # Create test dataloader using the first config
        val_loader, class_names = get_test_loader(configs[0])

        # Map class names to label IDs using the config label map
        label2id = {k: v for k, v in configs[0].train.label_map.items()}

        # Only test on classes relevant for the current dataset
        relevant = [label2id[c] for c in class_names]

        # Store metrics for each model (for later averaging)
        metrics = {"auroc": [],
                   "cmap": [],
                   "top1_acc": []}

        # ---------------------------------------------------------
        # test each model on the test dataset
        # ---------------------------------------------------------
        for model, _ in zip(models, configs):
            # Run evaluation
            auroc, cmap, top1_acc = test(model, val_loader, relevant, device=device)

            # Store metrics
            metrics["auroc"].append(auroc)
            metrics["cmap"].append(cmap)
            metrics["top1_acc"].append(top1_acc)

        # ---------------------------------------------------------
        # Aggregate metrics across all checkpoints/models
        # ---------------------------------------------------------
        results[down_task] = {key: float(np.mean(values)) for key, values in metrics.items()}
        print(results)

## 1. Impact of incorporating secondary label annotations into training, comparing model evaluation metrics with and without annotation weighting on the HSN test set (Table 5)

In [10]:
# download/load checkpoints trained without secondary labels
if not os.path.exists('../ckpts/rest/HSN_secw0'):
    gdown.download_folder('https://drive.google.com/drive/folders/1LiteKuM42wP1YuWUIAaA-bqxtTXy1GBS?usp=sharing', output="../ckpts/rest/")

# get ckpts paths 
ckpts = glob.glob('../ckpts/rest/HSN_secw0/*')

run_testing(ckpts, device=DEVICE)
    

['../ckpts/rest/HSN_secw0/HSN_eca_nfnet_l1_2026-03-09_122507', '../ckpts/rest/HSN_secw0/HSN_eca_nfnet_l1_2026-03-09_124840', '../ckpts/rest/HSN_secw0/HSN_eca_nfnet_l1_2026-03-09_123657']


2026-05-04 18:10:39,386 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 18:10:39,652 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-04 18:10:39,656 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-04 18:10:40,006 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 18:10:40,124 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-04 18:10:40,136 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-04 18:10:40,561 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 18:10:40,688 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9502559753896435, 'cmap': 0.5623522065687582, 'top1_acc': 0.7183063626289368}}


In [11]:
if not os.path.exists('../ckpts/DT/HSN'):
        gdown.download_folder('https://drive.google.com/drive/folders/1pgdC-y47iGGx6iVe5QuCC_kMiT7wkACO?usp=sharing', output="../ckpts/DT/")

# with secondary labels 
ckpts = glob.glob('../ckpts/DT/HSN/*')
run_testing(ckpts, device=DEVICE)

Retrieving folder contents


Retrieving folder 183re4OR2HRTG6ht6u0QS8ZvSxAjGgRQS HSN_eca_nfnet_l1_2025-10-20_112131
Retrieving folder 1vsZZrY7NyiocNV7LPF9Sj1ll3snK3jcQ models
Processing file 15S14s09RT0FL3kWS_E8vSDL3vAJwQ2LN model.pth
Retrieving folder 17iLiMA-gBFHc1Ar5Q6vwijdCaSrPODSf HSN_eca_nfnet_l1_2025-10-20_113316
Retrieving folder 1eHgBd0prTETs9B__PFV-CJsSGvnys6-V models
Processing file 13i_GPCrDNKhSZ0r7lqEzpH1D_SEhN9NL model.pth
Retrieving folder 1H4zFjeKtVawQEsaEOkn9YkoPyLvqxtFJ HSN_eca_nfnet_l1_2025-10-20_114501
Retrieving folder 1QyOgPYnyq_nclrzhO2HcAzLlN3vBhkFl models
Processing file 1B1ybhhGKTFFnOhh599hDzIZXAa_kvAwM model.pth


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=15S14s09RT0FL3kWS_E8vSDL3vAJwQ2LN
From (redirected): https://drive.google.com/uc?id=15S14s09RT0FL3kWS_E8vSDL3vAJwQ2LN&confirm=t&uuid=74ad6089-cd7e-44cb-ae7e-7b83b9e7b638
To: /home/bellafkir/Documents/sa4birds-main/ckpts/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_112131/models/model.pth
100%|██████████| 230M/230M [00:47<00:00, 4.81MB/s] 
Downloading...
From (original): https://drive.google.com/uc?id=13i_GPCrDNKhSZ0r7lqEzpH1D_SEhN9NL
From (redirected): https://drive.google.com/uc?id=13i_GPCrDNKhSZ0r7lqEzpH1D_SEhN9NL&confirm=t&uuid=3ae06365-d935-4431-9f76-483bfb1a3182
To: /home/bellafkir/Documents/sa4birds-main/ckpts/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_113316/models/model.pth
100%|██████████| 230M/230M [00:09<00:00, 24.9MB/s] 
Downloading...
From (original): https://drive.google.com/uc?id=1B1ybhhGKTFFnOhh599hDzIZXAa_kvAwM
From (redi

['../ckpts/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_113316', '../ckpts/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_112131', '../ckpts/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_114501']


2026-05-04 18:14:35,107 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 18:14:35,230 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-04 18:14:35,243 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-04 18:14:35,614 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 18:14:35,740 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-04 18:14:35,754 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-04 18:14:36,111 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 18:14:36,230 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9560262623085944, 'cmap': 0.5792627376801449, 'top1_acc': 0.7324406305948893}}


## 2. Performance metrics for various classifier heads tested on the HSN (Table 4)

In [12]:
# DSA 
ckpts = glob.glob('../ckpts/DT/HSN/*')
run_testing(ckpts, device=DEVICE)

['../ckpts/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_113316', '../ckpts/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_112131', '../ckpts/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_114501']


2026-05-04 18:17:53,847 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 18:17:53,963 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-04 18:17:53,978 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-04 18:17:54,317 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 18:17:54,428 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-04 18:17:54,443 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-04 18:17:54,953 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 18:17:55,068 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9560262623085944, 'cmap': 0.5792627376801449, 'top1_acc': 0.7324406305948893}}


In [13]:
# Linear

if not os.path.exists('../ckpts/rest/HSN_linearhead'):
    gdown.download_folder('https://drive.google.com/drive/folders/1zun4_eNfs_l0aXKWPR-ujmTy7LHsc5k5?usp=sharing', output="../ckpts/rest/")


ckpts = glob.glob('../ckpts/rest/HSN_linearhead/*')
run_testing(ckpts, device=DEVICE)

['../ckpts/rest/HSN_linearhead/HSN_eca_nfnet_l1_2026-03-09_153017', '../ckpts/rest/HSN_linearhead/HSN_eca_nfnet_l1_2026-03-09_151901', '../ckpts/rest/HSN_linearhead/HSN_eca_nfnet_l1_2026-03-09_150746']


2026-05-04 18:20:24,001 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 18:20:24,120 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-04 18:20:24,124 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-04 18:20:24,422 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 18:20:24,534 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-04 18:20:24,547 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-04 18:20:24,827 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 18:20:24,942 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9120344598123685, 'cmap': 0.5234318648468855, 'top1_acc': 0.6881780425707499}}


In [14]:
run_testing(ckpts, extracted_interval=5, target_length=501, device=DEVICE)

['../ckpts/rest/HSN_linearhead/HSN_eca_nfnet_l1_2026-03-09_153017', '../ckpts/rest/HSN_linearhead/HSN_eca_nfnet_l1_2026-03-09_151901', '../ckpts/rest/HSN_linearhead/HSN_eca_nfnet_l1_2026-03-09_150746']


2026-05-04 18:22:45,604 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 18:22:45,722 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-04 18:22:45,725 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-04 18:22:46,155 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 18:22:46,276 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-04 18:22:46,290 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-04 18:22:46,762 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 18:22:46,879 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9261919057612878, 'cmap': 0.5521745045732435, 'top1_acc': 0.6725559631983439}}


In [15]:
# Time Attention
if not os.path.exists('../ckpts/rest/HSN_timeattention'):
    gdown.download_folder('https://drive.google.com/drive/folders/1NekF8AcJWMmKqiuYDrJoA2SewlqjmATu?usp=sharing', output="../ckpts/rest/")

ckpts = glob.glob('../ckpts/rest/HSN_timeattention/*')


run_testing(ckpts, device=DEVICE)

Retrieving folder contents


Retrieving folder 15SDes17j3zmTuL29Y09UAkU9SzcNy8Es HSN_eca_nfnet_l1_2026-03-09_161144
Retrieving folder 1AwAh0Gr3af9uSM3v6SXExRK588ocqDwS models
Processing file 1DJFb1AW3hrUNrXW4zm9ns__zVB8D91-A model.pth
Retrieving folder 1KOLCnGQE4o8zsIBNvX1yiPyS84gXSFDV HSN_eca_nfnet_l1_2026-03-09_162312
Retrieving folder 1FDRZUZf3GCaDSj7qmp860gWCGft_QUwI models
Processing file 1cjhVqNc4oVYkZNsgnliVnQcQ-D4pKNYF model.pth
Retrieving folder 1s3bI-awtT2WBS3QPn94iueT9YtuyxJBm HSN_eca_nfnet_l1_2026-03-09_163440
Retrieving folder 15f72cqXZPG2dFROrj_PkUPazazs605vj models
Processing file 1C_sM44ywoNUz0xIGR8kQwfMXZ0Ho1DO4 model.pth


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1DJFb1AW3hrUNrXW4zm9ns__zVB8D91-A
From (redirected): https://drive.google.com/uc?id=1DJFb1AW3hrUNrXW4zm9ns__zVB8D91-A&confirm=t&uuid=7854a182-76de-4966-81be-1056b3275d9a
To: /home/bellafkir/Documents/sa4birds-main/ckpts/rest/HSN_timeattention/HSN_eca_nfnet_l1_2026-03-09_161144/models/model.pth
100%|██████████| 192M/192M [00:01<00:00, 98.3MB/s] 
Downloading...
From (original): https://drive.google.com/uc?id=1cjhVqNc4oVYkZNsgnliVnQcQ-D4pKNYF
From (redirected): https://drive.google.com/uc?id=1cjhVqNc4oVYkZNsgnliVnQcQ-D4pKNYF&confirm=t&uuid=63ab31a0-6a72-4798-80b8-1b0324ababbc
To: /home/bellafkir/Documents/sa4birds-main/ckpts/rest/HSN_timeattention/HSN_eca_nfnet_l1_2026-03-09_162312/models/model.pth
100%|██████████| 192M/192M [00:40<00:00, 4.76MB/s] 
Downloading...
From (original): https://drive.google.com/uc?id=1C_sM44ywoNU

['../ckpts/rest/HSN_timeattention/HSN_eca_nfnet_l1_2026-03-09_162312', '../ckpts/rest/HSN_timeattention/HSN_eca_nfnet_l1_2026-03-09_161144', '../ckpts/rest/HSN_timeattention/HSN_eca_nfnet_l1_2026-03-09_163440']


2026-05-04 18:25:26,895 | INFO | Task: HSN | Number of events: 12000
Testing: 100%|██████████| 94/94 [00:46<00:00,  2.04it/s]


{'HSN': {'auroc': 0.9457022326757826, 'cmap': 0.5847186219686932, 'top1_acc': 0.7046060164769491}}


In [16]:
# SSA
if not os.path.exists('../ckpts/rest/HSN_SSA'):
    gdown.download_folder('https://drive.google.com/drive/folders/1uxiSKvuclFVgqn1_oVvZZixUfKRnBE0f?usp=sharing', output="../ckpts/rest/")

ckpts = glob.glob('../ckpts/rest/HSN_SSA/*')


run_testing(ckpts, device=DEVICE)

Retrieving folder contents


Retrieving folder 11qRm0eBHT3eVoIWFvxkdY9XaSAFBfsjI HSN_eca_nfnet_l1_2026-03-14_102758
Retrieving folder 1wBmjM-eFYlHeOyxFC1Lel9BDoGh6CbXP models
Processing file 1TwKaMBBmwgr51Q2Vx03NZznu9YgF_oJR model.pth
Retrieving folder 1vvX9O91-_Hmrnvm-PWLR0Lu0LuL-e5TS HSN_eca_nfnet_l1_2026-03-14_103929
Retrieving folder 1oIsbBqd7EhRxKGA_9Ae6bW8vAK7IBM0W models
Processing file 1LxrSgSAbeOn1gmNI3LLoQByHAb0cOaR0 model.pth
Retrieving folder 1taxllrw359iaGWFXSrQ3orUQWy83lFUk HSN_eca_nfnet_l1_2026-03-14_105101
Retrieving folder 1nCgIH54CGF92_GYr8NG9Ra9xQKZEv8Q2 models
Processing file 1m3N8YEnBrc5TlS0gEi5JQkMx_cPpgeV- model.pth


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1TwKaMBBmwgr51Q2Vx03NZznu9YgF_oJR
From (redirected): https://drive.google.com/uc?id=1TwKaMBBmwgr51Q2Vx03NZznu9YgF_oJR&confirm=t&uuid=b5b0d823-2468-4535-8268-b27a28993aec
To: /home/bellafkir/Documents/sa4birds-main/ckpts/rest/HSN_SSA/HSN_eca_nfnet_l1_2026-03-14_102758/models/model.pth
100%|██████████| 192M/192M [00:03<00:00, 50.8MB/s] 
Downloading...
From (original): https://drive.google.com/uc?id=1LxrSgSAbeOn1gmNI3LLoQByHAb0cOaR0
From (redirected): https://drive.google.com/uc?id=1LxrSgSAbeOn1gmNI3LLoQByHAb0cOaR0&confirm=t&uuid=3bf14696-f4c7-4952-8038-2eec3e086382
To: /home/bellafkir/Documents/sa4birds-main/ckpts/rest/HSN_SSA/HSN_eca_nfnet_l1_2026-03-14_103929/models/model.pth
100%|██████████| 192M/192M [00:03<00:00, 52.8MB/s] 
Downloading...
From (original): https://drive.google.com/uc?id=1m3N8YEnBrc5TlS0gEi5JQkMx_cPpgeV

['../ckpts/rest/HSN_SSA/HSN_eca_nfnet_l1_2026-03-14_105101', '../ckpts/rest/HSN_SSA/HSN_eca_nfnet_l1_2026-03-14_102758', '../ckpts/rest/HSN_SSA/HSN_eca_nfnet_l1_2026-03-14_103929']


2026-05-04 18:28:09,800 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 18:28:09,916 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-04 18:28:09,930 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-04 18:28:10,374 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 18:28:10,489 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-04 18:28:10,492 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-04 18:28:10,839 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 18:28:10,958 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9551479085393016, 'cmap': 0.5842041371826746, 'top1_acc': 0.724009652932485}}


## 3. Effect of temporal context length during testing, showing how segment extension enhances model performance (Table 6)

In [17]:
# DFA using a 7-second input, where the centered 5 seconds are the target samples.
# target_length is the time resolution of the spectrograms
ckpts = glob.glob('../ckpts/DT/HSN/*')
run_testing(ckpts, extracted_interval=7, target_length=701, device=DEVICE)

['../ckpts/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_113316', '../ckpts/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_112131', '../ckpts/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_114501']


2026-05-04 18:30:37,838 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 18:30:37,987 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-04 18:30:38,001 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-04 18:30:38,363 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 18:30:38,490 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-04 18:30:38,503 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-04 18:30:38,885 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 18:30:39,011 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9560262623085944, 'cmap': 0.5792627376801449, 'top1_acc': 0.7324406305948893}}


In [18]:
# DFA using the original 5-second input of the test samples.
run_testing(ckpts, extracted_interval=5, target_length=501, device=DEVICE)

['../ckpts/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_113316', '../ckpts/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_112131', '../ckpts/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_114501']


2026-05-04 18:33:03,288 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 18:33:03,406 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-04 18:33:03,420 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-04 18:33:03,831 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 18:33:03,947 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-04 18:33:03,960 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-04 18:33:04,341 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 18:33:04,456 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9524781767262779, 'cmap': 0.5791612980916198, 'top1_acc': 0.7189263105392456}}


## 4. Ablation study on data augmentation methods, evaluating their impact on bird species recognition using HSN test data (Table 7)

In [19]:
# Baseline: No Augmentation

if not os.path.exists('../ckpts/rest/HSN_noaug'):
    gdown.download_folder('https://drive.google.com/drive/folders/1o6F1FxLb4vrBle1TU-U41x42QQx756ge?usp=sharing', output="../ckpts/rest/")


ckpts = glob.glob('../ckpts/rest/HSN_noaug/*')
run_testing(ckpts, device=DEVICE)

Retrieving folder contents


Retrieving folder 1BjcMSJe44xUeWYhaFRGoCTtvvi1qxmef HSN_eca_nfnet_l1_2025-10-27_083456
Retrieving folder 1ALf5XaIAjYBBVX7cS8X7XPwpp-_qyvL7 models
Processing file 1HiS6IrSx3Ig_MW_qNnXr0Cc4PL4kqoTJ model.pth
Retrieving folder 1RSn5eTggp6gs3uDv3withBUYwELAildS HSN_eca_nfnet_l1_2025-10-27_084621
Retrieving folder 1gZ74_pWus1NlCnJ8LyfhZDwaTeq230KD models
Processing file 1w4oe6TlHcDs4eeeYU_LokccHJWn3ZX9e model.pth
Retrieving folder 1uBkclQH7Xh8wpGhqMaanTvPgiVW_kOPq HSN_eca_nfnet_l1_2025-10-27_085747
Retrieving folder 1XDOAKU_sWEUDAM9EJVFgigkqF_VN1rUK models
Processing file 1ogjt6kQLIdiaITrNgHoUWHL5X6ULmybs model.pth


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1HiS6IrSx3Ig_MW_qNnXr0Cc4PL4kqoTJ
From (redirected): https://drive.google.com/uc?id=1HiS6IrSx3Ig_MW_qNnXr0Cc4PL4kqoTJ&confirm=t&uuid=1fa57185-41bf-44c0-90d3-a1d1190dce20
To: /home/bellafkir/Documents/sa4birds-main/ckpts/rest/HSN_noaug/HSN_eca_nfnet_l1_2025-10-27_083456/models/model.pth
100%|██████████| 230M/230M [00:04<00:00, 52.3MB/s] 
Downloading...
From (original): https://drive.google.com/uc?id=1w4oe6TlHcDs4eeeYU_LokccHJWn3ZX9e
From (redirected): https://drive.google.com/uc?id=1w4oe6TlHcDs4eeeYU_LokccHJWn3ZX9e&confirm=t&uuid=11fb5c45-6677-438c-b4b9-a7c2f6b069a0
To: /home/bellafkir/Documents/sa4birds-main/ckpts/rest/HSN_noaug/HSN_eca_nfnet_l1_2025-10-27_084621/models/model.pth
100%|██████████| 230M/230M [00:10<00:00, 21.4MB/s] 
Downloading...
From (original): https://drive.google.com/uc?id=1ogjt6kQLIdiaITrNgHoUWHL5X6U

['../ckpts/rest/HSN_noaug/HSN_eca_nfnet_l1_2025-10-27_084621', '../ckpts/rest/HSN_noaug/HSN_eca_nfnet_l1_2025-10-27_083456', '../ckpts/rest/HSN_noaug/HSN_eca_nfnet_l1_2025-10-27_085747']


2026-05-04 18:35:24,642 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 18:35:24,758 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-04 18:35:24,766 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-04 18:35:25,158 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 18:35:25,271 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-04 18:35:25,285 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-04 18:35:25,610 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 18:35:25,726 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.8588290967953124, 'cmap': 0.5249688989157181, 'top1_acc': 0.6417457063992819}}


In [20]:
# +Mixup (Signal)

if not os.path.exists('../ckpts/rest/HSN_sigmixup'):
    gdown.download_folder('https://drive.google.com/drive/folders/1hBOIo1XBc94B1penreZcFqWsMwsVlLJ1?usp=sharing', output="../ckpts/rest/")


ckpts = glob.glob('../ckpts/rest/HSN_sigmixup/*')
run_testing(ckpts, device=DEVICE)

Retrieving folder contents


Retrieving folder 1bCIZqZeKTfaJkUYv9Y7d9ivOv3vvKTqj HSN_eca_nfnet_l1_2025-10-27_095651
Retrieving folder 1ByPZfP28Hu6rnFr-RHQDMTC5WMc4Qmc9 models
Processing file 1GCIA86oBNVOBIHvUpkfECBxeScX60CBV model.pth
Retrieving folder 1nVpiOBCdKYEpMrexb98YDk0o2b8x5oOg HSN_eca_nfnet_l1_2025-10-27_100822
Retrieving folder 1qInDZCh3EONgDsZDtk8Z9xE7EnRXdfwo models
Processing file 11AHuOz1Zs-ZiXFYXbgwjlZZTm_jPFzcN model.pth
Retrieving folder 1tDptWga3WbQe4gon4q0rZ3zKSHkAbt_0 HSN_eca_nfnet_l1_2025-10-27_101953
Retrieving folder 1KEi2ejtC0Ja5ETJfEViOhSC3q9-UeHJZ models
Processing file 1INdHVJbHBqOx7JU5FIaLxvQ1Fybh3n-j model.pth


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1GCIA86oBNVOBIHvUpkfECBxeScX60CBV
From (redirected): https://drive.google.com/uc?id=1GCIA86oBNVOBIHvUpkfECBxeScX60CBV&confirm=t&uuid=bd983acd-6930-4286-b8a6-73fb0d4c859c
To: /home/bellafkir/Documents/sa4birds-main/ckpts/rest/HSN_sigmixup/HSN_eca_nfnet_l1_2025-10-27_095651/models/model.pth
100%|██████████| 230M/230M [00:06<00:00, 38.1MB/s] 
Downloading...
From (original): https://drive.google.com/uc?id=11AHuOz1Zs-ZiXFYXbgwjlZZTm_jPFzcN
From (redirected): https://drive.google.com/uc?id=11AHuOz1Zs-ZiXFYXbgwjlZZTm_jPFzcN&confirm=t&uuid=7c8ebead-ee4f-4dca-8589-88fd1b1c4884
To: /home/bellafkir/Documents/sa4birds-main/ckpts/rest/HSN_sigmixup/HSN_eca_nfnet_l1_2025-10-27_100822/models/model.pth
100%|██████████| 230M/230M [00:02<00:00, 78.5MB/s] 
Downloading...
From (original): https://drive.google.com/uc?id=1INdHVJbHBqOx7JU5FIaLx

['../ckpts/rest/HSN_sigmixup/HSN_eca_nfnet_l1_2025-10-27_100822', '../ckpts/rest/HSN_sigmixup/HSN_eca_nfnet_l1_2025-10-27_095651', '../ckpts/rest/HSN_sigmixup/HSN_eca_nfnet_l1_2025-10-27_101953']


2026-05-04 18:38:18,279 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 18:38:18,392 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-04 18:38:18,406 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-04 18:38:18,792 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 18:38:18,909 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-04 18:38:18,919 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-04 18:38:19,305 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 18:38:19,415 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9059694394473373, 'cmap': 0.5447450570350201, 'top1_acc': 0.7146488030751547}}


In [21]:
# +Mixup (Signal) +Background-noise

if not os.path.exists('../ckpts/rest/HSN_sigmixup+bgnoise'):
    gdown.download_folder('https://drive.google.com/drive/folders/12mt86SVO8yDIpRSjhSNLzrqq_JT4EaAN?usp=sharing', output="../ckpts/rest/")


ckpts = glob.glob('../ckpts/rest/HSN_sigmixup+bgnoise/*')
run_testing(ckpts, device=DEVICE)

Retrieving folder contents


Retrieving folder 1jJhEgrYBKhfw1B3RPzmLbewDbC2yVogE HSN_eca_nfnet_l1_2025-10-27_103302
Retrieving folder 1JF4vFfdLU4cZgQl1GBsy29cez8at0axm models
Processing file 1JYpkOQg-BTgzwErRuKH_EWZNUS7yxEMy model.pth
Retrieving folder 1ZzNpvpnptdOxksmHuYgul6z0Ev9COxW7 HSN_eca_nfnet_l1_2025-10-27_104438
Retrieving folder 1gkXWRsYBa-S3hbOzb7hhcyDG1joiQ-s5 models
Processing file 18xmdlWfopVyHTfkoKGuroEcKGc1fuIUp model.pth
Retrieving folder 1x1J-DWKWlWJmMn3e6JNyYvVkVSVnS6bY HSN_eca_nfnet_l1_2025-10-27_105614
Retrieving folder 1JkEopjXL1lkj7xbMT9by9CcB8fMsWMg6 models
Processing file 15q1xCiRxLg3N8f2wwKxpaR5wg_DkYFih model.pth


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1JYpkOQg-BTgzwErRuKH_EWZNUS7yxEMy
From (redirected): https://drive.google.com/uc?id=1JYpkOQg-BTgzwErRuKH_EWZNUS7yxEMy&confirm=t&uuid=d1b1ad07-db79-42f8-8448-423ed97cd913
To: /home/bellafkir/Documents/sa4birds-main/ckpts/rest/HSN_sigmixup+bgnoise/HSN_eca_nfnet_l1_2025-10-27_103302/models/model.pth
100%|██████████| 230M/230M [00:03<00:00, 69.6MB/s] 
Downloading...
From (original): https://drive.google.com/uc?id=18xmdlWfopVyHTfkoKGuroEcKGc1fuIUp
From (redirected): https://drive.google.com/uc?id=18xmdlWfopVyHTfkoKGuroEcKGc1fuIUp&confirm=t&uuid=6c3741ec-3b34-4e94-b4be-7f6f9980245e
To: /home/bellafkir/Documents/sa4birds-main/ckpts/rest/HSN_sigmixup+bgnoise/HSN_eca_nfnet_l1_2025-10-27_104438/models/model.pth
100%|██████████| 230M/230M [00:04<00:00, 56.4MB/s] 
Downloading...
From (original): https://drive.google.com/uc?id=15q1xC

['../ckpts/rest/HSN_sigmixup+bgnoise/HSN_eca_nfnet_l1_2025-10-27_103302', '../ckpts/rest/HSN_sigmixup+bgnoise/HSN_eca_nfnet_l1_2025-10-27_105614', '../ckpts/rest/HSN_sigmixup+bgnoise/HSN_eca_nfnet_l1_2025-10-27_104438']


2026-05-04 18:41:16,792 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 18:41:16,910 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-04 18:41:16,923 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-04 18:41:17,299 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 18:41:17,413 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-04 18:41:17,427 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-04 18:41:17,858 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 18:41:17,976 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9062293678286514, 'cmap': 0.5517955212866806, 'top1_acc': 0.7202281554539999}}


In [22]:
# +Mixup (Signal) +Background noise +colored noise +gain

if not os.path.exists('../ckpts/rest/HSN_sigmixup+bgnoise+colorednoise+gain'):
    gdown.download_folder('https://drive.google.com/drive/folders/13Sy9zOTHx1PPkhesRaQnb-gSYQMvbE0L?usp=sharing', output="../ckpts/rest/")


ckpts = glob.glob('../ckpts/rest/HSN_sigmixup+bgnoise+colorednoise+gain/*')
run_testing(ckpts, device=DEVICE)

Retrieving folder contents


Retrieving folder 11iZiOOIIdO4wZuEO67eBHHVu77MM11Uu HSN_eca_nfnet_l1_2026-03-09_203948
Retrieving folder 1Vw5J30OTbOesjmAqj-rE1Q-iFj8LqNU1 models
Processing file 1l51D7oo62rZ4pMGe06Uog3U1gYYtOMjy model.pth
Retrieving folder 1YzLguLjIzIyiWnvqc9WDTwk3iNu_aZUX HSN_eca_nfnet_l1_2026-03-09_205253
Retrieving folder 1fUuepXBeP-mGafAmRFIvLTD3RVD5drjQ models
Processing file 1NnrO3ka5yG8oUZYBZzk6WsVAFlOR8lVk model.pth
Retrieving folder 1uGPBjIJ_tI3HfuEgP1rZQxd5XEk8ogNd HSN_eca_nfnet_l1_2026-03-09_210951
Retrieving folder 1O7VqI_3hOCRnZ4Ws315itRmhbHOL8cZP models
Processing file 1Wqb2_rBb9FwQ-DZMbVWgKrVleCbr9OZ8 model.pth


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1l51D7oo62rZ4pMGe06Uog3U1gYYtOMjy
From (redirected): https://drive.google.com/uc?id=1l51D7oo62rZ4pMGe06Uog3U1gYYtOMjy&confirm=t&uuid=93c16c70-6872-449a-8bf4-8df66834f5ae
To: /home/bellafkir/Documents/sa4birds-main/ckpts/rest/HSN_sigmixup+bgnoise+colorednoise+gain/HSN_eca_nfnet_l1_2026-03-09_203948/models/model.pth
100%|██████████| 230M/230M [00:02<00:00, 88.2MB/s] 
Downloading...
From (original): https://drive.google.com/uc?id=1NnrO3ka5yG8oUZYBZzk6WsVAFlOR8lVk
From (redirected): https://drive.google.com/uc?id=1NnrO3ka5yG8oUZYBZzk6WsVAFlOR8lVk&confirm=t&uuid=62962aeb-df36-4162-8e7f-c9b9eb924e05
To: /home/bellafkir/Documents/sa4birds-main/ckpts/rest/HSN_sigmixup+bgnoise+colorednoise+gain/HSN_eca_nfnet_l1_2026-03-09_205253/models/model.pth
100%|██████████| 230M/230M [00:06<00:00, 37.1MB/s] 
Downloading...
From (original): h

['../ckpts/rest/HSN_sigmixup+bgnoise+colorednoise+gain/HSN_eca_nfnet_l1_2026-03-09_205253', '../ckpts/rest/HSN_sigmixup+bgnoise+colorednoise+gain/HSN_eca_nfnet_l1_2026-03-09_203948', '../ckpts/rest/HSN_sigmixup+bgnoise+colorednoise+gain/HSN_eca_nfnet_l1_2026-03-09_210951']


2026-05-04 18:44:10,696 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 18:44:10,834 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-04 18:44:10,841 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-04 18:44:11,188 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 18:44:11,304 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-04 18:44:11,319 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-04 18:44:11,733 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 18:44:11,848 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9115621064078656, 'cmap': 0.5518943480274384, 'top1_acc': 0.7261794209480286}}


In [23]:
# +Mixup (Signal) +Background-noise +colored-noise +gain +no-call 

if not os.path.exists('../ckpts/rest/HSN_sigmixup+bgnoise+colorednoise+gain+nocall'):
    gdown.download_folder('https://drive.google.com/drive/folders/1I7Bpd1hDMxyoaqdGyM5Nqn0onxUnZtce?usp=sharing', output="../ckpts/rest/")


ckpts = glob.glob('../ckpts/rest/HSN_sigmixup+bgnoise+colorednoise+gain+nocall/*')
run_testing(ckpts, device=DEVICE)

Retrieving folder contents


Retrieving folder 1JLALb5hEeOT14auoTNsX_eCXpC0UocZU HSN_eca_nfnet_l1_2026-03-10_113228
Retrieving folder 1q3tM7NSAuy2I93rngKsmCUPr5ySmG5Ne models
Processing file 11kDnBttFu8IP1o22_sjhLh24z3KDIe1g model.pth
Retrieving folder 1RLhOcIAbOlCmok_DeOG13ETOe97O-Aej HSN_eca_nfnet_l1_2026-03-10_114410
Retrieving folder 1hJvpOiddyEirt7UIc4dPFXrUPL8raGpO models
Processing file 1KRF8Ohh9OpP0-Uef8Vwrdb1kgVBisATT model.pth
Retrieving folder 1KWyeevdnAM3yWbvLh0XOypjuO7rigQUW HSN_eca_nfnet_l1_2026-03-10_115551
Retrieving folder 1oNN6ivP9yVpwtKF1PIt_LFRieERtYhY6 models
Processing file 1IdIPT7x0sDmeXthN_n_TQs00yRr4OsWS model.pth


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=11kDnBttFu8IP1o22_sjhLh24z3KDIe1g
From (redirected): https://drive.google.com/uc?id=11kDnBttFu8IP1o22_sjhLh24z3KDIe1g&confirm=t&uuid=ac5e3d44-9f77-4a36-8336-ecdd78ab1d8f
To: /home/bellafkir/Documents/sa4birds-main/ckpts/rest/HSN_sigmixup+bgnoise+colorednoise+gain+nocall/HSN_eca_nfnet_l1_2026-03-10_113228/models/model.pth
100%|██████████| 230M/230M [00:02<00:00, 103MB/s]  
Downloading...
From (original): https://drive.google.com/uc?id=1KRF8Ohh9OpP0-Uef8Vwrdb1kgVBisATT
From (redirected): https://drive.google.com/uc?id=1KRF8Ohh9OpP0-Uef8Vwrdb1kgVBisATT&confirm=t&uuid=d39ddf36-1fb2-41c6-986f-81725bf2d4cd
To: /home/bellafkir/Documents/sa4birds-main/ckpts/rest/HSN_sigmixup+bgnoise+colorednoise+gain+nocall/HSN_eca_nfnet_l1_2026-03-10_114410/models/model.pth
100%|██████████| 230M/230M [00:11<00:00, 19.8MB/s] 
Downloading...
From

['../ckpts/rest/HSN_sigmixup+bgnoise+colorednoise+gain+nocall/HSN_eca_nfnet_l1_2026-03-10_113228', '../ckpts/rest/HSN_sigmixup+bgnoise+colorednoise+gain+nocall/HSN_eca_nfnet_l1_2026-03-10_115551', '../ckpts/rest/HSN_sigmixup+bgnoise+colorednoise+gain+nocall/HSN_eca_nfnet_l1_2026-03-10_114410']


2026-05-04 18:47:05,220 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 18:47:05,336 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-04 18:47:05,348 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-04 18:47:05,723 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 18:47:05,840 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-04 18:47:05,851 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-04 18:47:06,239 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-04 18:47:06,355 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9506547736325318, 'cmap': 0.5801166357128348, 'top1_acc': 0.7243196368217468}}


In [19]:
# +Mixup (Signal) +Background-noise +colored-noise +gain +no-call +Mixup (Spec) +SpecAug

ckpts = glob.glob('../ckpts/DT/HSN/*')
run_testing(ckpts, device=DEVICE)

['../checkpoints/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_113316', '../checkpoints/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_112131', '../checkpoints/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_114501']


2026-03-14 11:08:35,768 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 11:08:35,886 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 11:08:35,892 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 11:08:36,260 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 11:08:36,377 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-03-14 11:08:36,391 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-03-14 11:08:36,752 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-03-14 11:08:36,871 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9560262623085944, 'cmap': 0.5792627376801449, 'top1_acc': 0.7324406305948893}}
